# Chapter 01: N-gram Language Models

**Goal**: Build the simplest possible language model — one that predicts the next character based on the previous 1 or 2 characters.

**What you'll learn**:
- How bigrams and trigrams work (counting co-occurrences)
- Why local statistics are not enough for real tasks
- The first glimpse of the gap between pattern matching and understanding

**Human lens**: Humans don't count co-occurrences. They understand meaning, apply algorithms, and know when they don't have enough information.

## How N-grams Work

- **Bigram**: P(next | previous) — "what character usually follows this one?"
- **Trigram**: P(next | prev2, prev1) — "what character follows this pair?"

No neural network, no gradients — just counting.

In [1]:
from not_a_brain.models.tokenizer import CharTokenizer
from not_a_brain.models.ngram import BigramModel, TrigramModel, NgramAgent
from not_a_brain.tasks import (
    ArithmeticTask, CopyTask, GrammarTask,
    KnowledgeQATask, CompositionalTask, UnknownTask,
)

# Build training corpus from task data
tasks_train = {
    "arithmetic": ArithmeticTask(seed=1),
    "copy": CopyTask(seed=1),
    "grammar": GrammarTask(seed=1),
    "knowledge_qa": KnowledgeQATask(seed=1),
    "compositional": CompositionalTask(seed=1),
    "unknown": UnknownTask(seed=1),
}

corpus = []
for task in tasks_train.values():
    pairs = task.training_pairs(500)
    for prompt, answer in pairs:
        corpus.append(prompt + answer)

print(f"Training corpus: {len(corpus)} sequences")
print(f"Example: '{corpus[0][:60]}...'")

Training corpus: 3000 sequences
Example: 'ADD 72 97 =169...'


In [2]:
# Fit tokenizer and train both models
tokenizer = CharTokenizer()
tokenizer.fit(corpus)
print(f"Vocab size: {tokenizer.vocab_size} characters")

bigram = BigramModel(tokenizer)
bigram.train(corpus)
print("Bigram model trained.")

trigram = TrigramModel(tokenizer)
trigram.train(corpus)
print("Trigram model trained.")

Vocab size: 79 characters


Bigram model trained.


Trigram model trained.


## Sample Generations

Let's see what these models produce. Watch how the bigram drifts quickly, while the trigram has slightly more coherence — but neither truly "understands" the task.

In [3]:
prompts = [
    ("Arithmetic", "ADD 5 3 ="),
    ("Copy", "COPY: abc|"),
    ("Grammar", "CHECK: ( )"),
    ("Knowledge QA", "FACT: paris is capital of france. Q: capital of france?"),
]

for label, prompt in prompts:
    bi_out = bigram.generate(prompt, max_len=20)
    tri_out = trigram.generate(prompt, max_len=20)
    print(f"[{label}]")
    print(f"  Prompt:  '{prompt}'")
    print(f"  Bigram:  '{bi_out}'")
    print(f"  Trigram: '{tri_out}'")
    print()

[Arithmetic]
  Prompt:  'ADD 5 3 ='
  Bigram:  'ADD 5 3 =19 of of of of of of'
  Trigram: 'ADD 5 3 =12'

[Copy]
  Prompt:  'COPY: abc|'
  Bigram:  'COPY: abc|r of of of of of of '
  Trigram: 'COPY: abc|szbg|wqvlid'

[Grammar]
  Prompt:  'CHECK: ( )'
  Bigram:  'CHECK: ( ) of of of of of of o'
  Trigram: 'CHECK: ( ) ] } ( ) ] } ( ) ] }'

[Knowledge QA]
  Prompt:  'FACT: paris is capital of france. Q: capital of france?'
  Bigram:  'FACT: paris is capital of france. Q: capital of france?un of of of of of of'
  Trigram: 'FACT: paris is capital of france. Q: capital of france?par of from the TO t'



## Evaluation: N-grams vs Human Agent

Now let's run the full eval suite and compare.

In [4]:
from not_a_brain.evals.harness import run_eval_suite
from not_a_brain.human_agent.agent import HumanAgent
from not_a_brain.utils.visualization import plot_comparison_bar

# Eval tasks (different seed = no data leakage)
eval_tasks = {
    "arithmetic": ArithmeticTask(seed=99),
    "copy": CopyTask(seed=99),
    "grammar": GrammarTask(seed=99),
    "knowledge_qa": KnowledgeQATask(seed=99),
    "compositional": CompositionalTask(seed=99),
    "unknown": UnknownTask(seed=99),
}

bigram_agent = NgramAgent(bigram, "bigram")
trigram_agent = NgramAgent(trigram, "trigram")
human = HumanAgent()

results = {}
for name, agent in [("bigram", bigram_agent), ("trigram", trigram_agent), ("human", human)]:
    metrics, _ = run_eval_suite(agent, eval_tasks, n_per_task=50)
    results[name] = metrics
    print(f"\n{name}:")
    print(f"  Accuracy:           {metrics.accuracy:.1%}")
    print(f"  Hallucination rate: {metrics.hallucination_rate:.1%}")
    print(f"  Abstention rate:    {metrics.abstention_rate:.1%}")
    print(f"  Per-task:")
    for t, info in metrics.per_task.items():
        print(f"    {t:20s} {info['accuracy']:.1%}")


bigram:
  Accuracy:           0.0%
  Hallucination rate: 100.0%
  Abstention rate:    0.0%
  Per-task:
    arithmetic           0.0%
    copy                 0.0%
    grammar              0.0%
    knowledge_qa         0.0%
    compositional        0.0%
    unknown              0.0%

trigram:
  Accuracy:           13.7%
  Hallucination rate: 18.0%
  Abstention rate:    0.0%
  Per-task:
    arithmetic           0.0%
    copy                 0.0%
    grammar              0.0%
    knowledge_qa         0.0%
    compositional        0.0%
    unknown              82.0%

human:
  Accuracy:           100.0%
  Hallucination rate: 0.0%
  Abstention rate:    16.7%
  Per-task:
    arithmetic           100.0%
    copy                 100.0%
    grammar              100.0%
    knowledge_qa         100.0%
    compositional        100.0%
    unknown              100.0%


In [5]:
# Visual comparison
task_names = list(eval_tasks.keys())
scores = {}
for name, metrics in results.items():
    scores[name] = [metrics.per_task.get(t, {}).get("accuracy", 0.0) for t in task_names]

fig = plot_comparison_bar(
    labels=task_names,
    scores=scores,
    title="Chapter 01: N-grams vs Human Agent",
    show=True,
)

C:\Users\delacruzribadenc\Documents\Repos\my_own_projects\not-a-brain\src\not_a_brain\utils\visualization.py:90: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## What to Observe

1. **Bigram gets ~0%** — knowing only the previous character is not enough for any task
2. **Trigram does slightly better** — it picks up some local patterns (e.g., grammar brackets sometimes repeat)
3. **Neither model abstains** — they always generate something, even for unanswerable questions (100% hallucination)
4. **The human agent gets 100%** — because it uses algorithms (stack for grammar, arithmetic rules) and abstains when uncertain

## Why N-grams Fail

- **No long-range dependencies**: `ADD 12 37 =` requires understanding the full prompt; a bigram only sees the `=`
- **No compositionality**: chained operations need tracking intermediate state
- **No abstention mechanism**: n-grams always produce the most likely next character — they can't say "I don't know"

## Human Lens

Humans don't solve arithmetic by finding the most common character after `=`. They:
1. Parse the numbers and operation
2. Apply a learned algorithm (addition)
3. Verify the result
4. Express uncertainty if the problem is unclear

This is fundamentally different from statistical co-occurrence.

## Next: Chapter 02 (Feed-Forward LM)

Adding a fixed-window MLP will let us capture slightly longer patterns — but it's still a far cry from understanding.